# Experiments

### Setup

In [ ]:
# You can set them inline
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"

In [ ]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

(do not use- use app.py in local)Here is the RAG Application that we've been working with throughout this course

In [ ]:
import os
import tempfile
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.sitemap import SitemapLoader
from langchain_community.vectorstores import SKLearnVectorStore
from langchain_openai import OpenAIEmbeddings
from langsmith import traceable
from openai import OpenAI
from typing import List
import nest_asyncio

# TODO: Configure this model!
MODEL_NAME = "gpt-4o"
MODEL_PROVIDER = "openai"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
"""

openai_client = OpenAI()

def get_vector_db_retriever():
    persist_path = os.path.join(tempfile.gettempdir(), "union.parquet")
    embd = OpenAIEmbeddings()

    # If vector store exists, then load it
    if os.path.exists(persist_path):
        vectorstore = SKLearnVectorStore(
            embedding=embd,
            persist_path=persist_path,
            serializer="parquet"
        )
        return vectorstore.as_retriever(lambda_mult=0)

    # Otherwise, index LangSmith documents and create new vector store
    ls_docs_sitemap_loader = SitemapLoader(web_path="https://docs.smith.langchain.com/sitemap.xml", continue_on_failure=True)
    ls_docs = ls_docs_sitemap_loader.load()

    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=500, chunk_overlap=0
    )
    doc_splits = text_splitter.split_documents(ls_docs)

    vectorstore = SKLearnVectorStore.from_documents(
        documents=doc_splits,
        embedding=embd,
        persist_path=persist_path,
        serializer="parquet"
    )
    vectorstore.persist()
    return vectorstore.as_retriever(lambda_mult=0)

nest_asyncio.apply()
retriever = get_vector_db_retriever()

"""
retrieve_documents
- Returns documents fetched from a vectorstore based on the user's question
"""
@traceable(run_type="chain")
def retrieve_documents(question: str):
    return retriever.invoke(question)

"""
generate_response
- Calls `call_openai` to generate a model response after formatting inputs
"""
@traceable(run_type="chain")
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Context: {formatted_docs} \n\n Question: {question}"
        }
    ]
    return call_openai(messages)

"""
call_openai
- Returns the chat completion output from OpenAI
"""
@traceable(
    run_type="llm",
    metadata={
        "ls_provider": MODEL_PROVIDER,
        "ls_model_name": MODEL_NAME
    }
)
def call_openai(messages: List[dict]) -> str:
    return openai_client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
    )

"""
langsmith_rag
- Calls `retrieve_documents` to fetch documents
- Calls `generate_response` to generate a response based on the fetched documents
- Returns the model response
"""
@traceable(run_type="chain")
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content


### Experiment

Here is a code snippet that should look similar to what you see from the starter code!

There are a few important components here.

1. We have defined an Evaluator
2. We pipe our dataset examples (dict) to the shape of input that our function `langsmith_rag` takes (str) using a target function

In [1]:
from langsmith import evaluate, Client
from app import langsmith_rag

client = Client()
dataset_name = "ds-rag-golden-dataset"

def is_concise_enough(reference_outputs: dict, outputs: dict) -> dict:
    score = len(outputs["output"]) < 1.5 * len(reference_outputs["output"])
    return {"key": "is_concise", "score": int(score)}

def target_function(inputs: dict):
    return langsmith_rag(inputs["question"])

evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="gpt-4o"
)

/home/harit/work/langchain_obs/intro-to-langsmith-main/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'gpt-4o-d2f1172a' at:
https://smith.langchain.com/o/9b4490db-7da0-47ba-a13b-0d5993ee5c60/datasets/89c19128-e2af-4f3b-bf1b-f077b8795b56/compare?selectedSessions=2bc88c13-449b-4254-a96b-69859f2f495c




0it [00:00, ?it/s]Error running evaluator <DynamicRunEvaluator is_concise_enough> on run dfd63a66-e5b9-4245-ab3d-b6fa79616e3f: KeyError('output')
Traceback (most recent call last):
  File "/home/harit/work/langchain_obs/intro-to-langsmith-main/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1619, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/harit/work/langchain_obs/intro-to-langsmith-main/.venv/lib/python3.12/site-packages/langsmith/evaluation/evaluator.py", line 351, in evaluate_run
    result = self.func(
             ^^^^^^^^^^
  File "/home/harit/work/langchain_obs/intro-to-langsmith-main/.venv/lib/python3.12/site-packages/langsmith/evaluation/evaluator.py", line 777, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_33579/3256492057.py", line 8, in is_concise_enough
    

,inputs.question,outputs.content,outputs.additional_kwargs,outputs.response_metadata,outputs.type,outputs.id,outputs.tool_calls,outputs.invalid_tool_calls,outputs.usage_metadata,error,...,example_id,id,reference.id,reference.type,reference.content,reference.tool_calls,reference.usage_metadata,reference.additional_kwargs,reference.response_metadata,reference.invalid_tool_calls
0,What testing capabilities does LangSmith have?,LangSmith offers evaluation and feedback colle...,{'refusal': None},"{'token_usage': {'completion_tokens': 51, 'pro...",ai,lc_run--c262eae9-056d-4d17-a7c8-c4e7972c9550-0,[],[],"{'input_tokens': 1067, 'output_tokens': 51, 't...",None,...,0243c54a-3762-4bc2-b56f-102ab0a61b09,dfd63a66-e5b9-4245-ab3d-b6fa79616e3f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Can LangSmith be used for finetuning and model...,I don't know.,{'refusal': None},"{'token_usage': {'completion_tokens': 5, 'prom...",ai,lc_run--8621e4e0-c184-44ac-99ce-baf4796dd53f-0,[],[],"{'input_tokens': 1072, 'output_tokens': 5, 'to...",None,...,34300280-0fb5-43c3-b4a3-20a4d206ac8c,95f8a013-af19-450f-b257-9783d3b0ffa4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,How do I set up tracing to LangSmith if I'm us...,To set up tracing to LangSmith while using Lan...,{'refusal': None},"{'token_usage': {'completion_tokens': 64, 'pro...",ai,lc_run--e86117f8-13d1-4898-9611-513917a9133f-0,[],[],"{'input_tokens': 1035, 'output_tokens': 64, 't...",None,...,4029ad78-299c-4d8a-85e4-38455877d47b,6481d5b3-3c36-46f1-a628-4843a0c04f7b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Can LangSmith be used to evaluate agents?,"Yes, LangSmith can be used to evaluate agents ...",{'refusal': None},"{'token_usage': {'completion_tokens': 51, 'pro...",ai,lc_run--bae61e0e-70f1-482f-85ba-559c247cb014-0,[],[],"{'input_tokens': 1068, 'output_tokens': 51, 't...",None,...,493d31fc-7012-42b8-817a-1271e80d817e,bee91c21-6032-4782-b643-1b3d748843d6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,How do I set up tracing to LangSmith if I'm us...,To set up tracing to LangSmith while using Lan...,{'refusal': None},"{'token_usage': {'completion_tokens': 67, 'pro...",ai,lc_run--90e12583-47a9-491e-9f81-8d84f01522b6-0,[],[],"{'input_tokens': 1035, 'output_tokens': 67, 't...",None,...,8b5ac1c2-98c1-421e-9b6b-504c287e8176,6d8fcc35-7f91-42a1-9f65-256f15e252ec,lc_run--f812278d-98ae-4a8d-846b-99f90d1b4e1c-0,ai,To set up tracing to LangSmith while using Lan...,[],"{'input_tokens': 1035, 'total_tokens': 1100, '...",{'refusal': None},{'id': 'chatcmpl-D6geO3hxTq8miOxs0S0vvX8EZhOvN...,[]
5,Does LangSmith support online evaluation?,"Yes, LangSmith supports online evaluation. You...",{'refusal': None},"{'token_usage': {'completion_tokens': 40, 'pro...",ai,lc_run--a48f1241-769a-4190-b80f-b0cd6fc33096-0,[],[],"{'input_tokens': 725, 'output_tokens': 40, 'to...",None,...,03b5b55a-ff35-4b05-922b-32cacbbb1f64,750f2f92-8ec3-4dc3-a51c-9bbf49ba6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,How do I create user feedback with the LangSmi...,To create user feedback with the LangSmith SDK...,{'refusal': None},"{'token_usage': {'completion_tokens': 62, 'pro...",ai,lc_run--fc56f073-b16a-4c7f-9f52-a885acb61704-0,[],[],"{'input_tokens': 1071, 'output_tokens': 62, 't...",None,...,1053a474-a68e-43ac-8911-1d6e245d8cd7,c5980f5f-44c8-4262-89d7-9025ba9322ec,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Does LangSmith support offline evaluation?,The provided context does not mention support ...,{'refusal': None},"{'token_usage': {'completion_tokens': 27, 'pro...",ai,lc_run--b9010a04-a554-43a2-9ea1-a44a4a98b556-0,[],[],"{'input_tokens': 725, 'output_tokens': 27, 'to...",None,...,1dfb5d52-a69b-46a9-b658-d857a09b1cd5,1ce0d5d6-4079-4dd3-a5a9-22363cd3b52d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,What is LangSmith used for in three sentences?,"LangSmith is used for observability, evaluatio...",{'refusal': None},"{'token_usage': {'completion_tokens': 50, 'pro...",ai,lc_run--402d8de1-cb5e-45f2-b4f5-4f889bfefa6d-0,[],[],"{'input_tokens': 1068, 'output_tokens': 50, 't...",None,...,30ab1714-4471-4588-b3fa-364d31dfb100,

### Modifying your Application

Now, let's change our model to gpt-35-turbo and see how it performs!

Make this change, and then run this code snippet!

In [ ]:
from langsmith import evaluate, Client
from langsmith.schemas import Example, Run

def target_function(inputs: dict):
    return langsmith_rag(inputs["question"])

evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="gpt-3.5-turbo"
)

### Running over Different pieces of Data

##### Dataset Version

You can execute an experiment on a specific version of a dataset in the sdk by using the `as_of` parameter in `list_examples`

Let's try running on just our initial dataset.

In [ ]:
evaluate(
    target_function,
    data=client.list_examples(dataset_name=dataset_name, as_of="initial dataset"),   # We use as_of to specify a version
    evaluators=[is_concise_enough],
    experiment_prefix="initial dataset version"
)

##### Dataset Split

You can run an experiment on a specific split of your dataset, let's try running on the Crucial Examples split.

In [ ]:
evaluate(
    target_function,
    data=client.list_examples(dataset_name=dataset_name, splits=["Crucial Examples"]),  # We pass in a list of Splits
    evaluators=[is_concise_enough],
    experiment_prefix="Crucial Examples split"
)

##### Specific Data Points

You can specify individual data points to run an experiment over as well

In [ ]:
evaluate(
    target_function,
    data=client.list_examples(
        dataset_name=dataset_name, 
        example_ids=[   # We pass in a specific list of example_ids
            # TODO: You will need to paste in your own example ids for this to work!
            "",
            ""
        ]
    ),
    evaluators=[is_concise_enough],
    experiment_prefix="two specific example ids"
)

### Other Parameters

##### Repetitions

You can run an experiment several times to make sure you have consistent results

In [ ]:
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="two repetitions",
    num_repetitions=2   # This field defaults to 1
)

##### Concurrency
You can also kick off concurrent threads of execution to make your experiments finish faster!

In [ ]:
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="concurrency",
    max_concurrency=3,  # This defaults to None, so this is an improvement!
)

##### Metadata 

You can (and should) add metadata to your experiments, to make them easier to find in the UI

In [ ]:
evaluate(
    target_function,
    data=dataset_name,
    evaluators=[is_concise_enough],
    experiment_prefix="metadata added",
    metadata={  # We can pass custom metadata for the experiment, such as the model name
        "model_name": MODEL_NAME
    }
)